# Overture — Building footprints (live)

Fetch building footprints for a neighbourhood block and plot them. Buildings is the largest theme (2.3 B rows globally), so a **bounded bbox is required** — the backend guards against an oversized box.

## Setup

Pull in the imports and the unified `EarthLens` entry point. `geopandas` reads the GeoParquet that the Overture backend writes to disk.

In [ ]:
import geopandas as gpd
from earthlens.earthlens import EarthLens

### The area of interest

Define a small bounding box over a Times Square block and the output directory. The box is given as `[south, north]` / `[west, east]` limits, which we assemble into the `[west, south, east, north]` `aoi` the backend expects.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

## Download the footprints

Build the Overture request first — the `buildings` dataset over the block, written as GeoParquet. Keeping the constructor on its own line makes the request easy to read and re-run.

In [ ]:
overture = EarthLens(
    data_source="overture",
    dataset="buildings",
    variables=[],
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
    file_format="geoparquet",
)

Now run the download. The vector backend writes one GeoParquet file and returns the list of written paths.

In [ ]:
paths = overture.download()
paths

### Inspect the result

Read the written GeoParquet back into a `GeoDataFrame` and check the geometry type and per-row `license_id` (Overture surfaces the source license on every feature).

In [ ]:
gdf = gpd.read_parquet(paths[0])
print(len(gdf), 'footprints;  geom', gdf.geometry.geom_type.unique())
gdf[['id', 'license_id']].head()

### Plot the footprints

A quick plot of the building polygons over the block:

In [ ]:
ax = gdf.plot(figsize=(5, 5), edgecolor='black', linewidth=0.3)
ax.set_title('Overture building footprints')
ax.set_axis_off()

## The bbox guard

The guard rejects a whole-Earth buildings request rather than trying to read billions of rows. We build the oversized request, then call `download()` inside a `try` so the raised `ValueError` is printed instead of aborting the notebook.

In [ ]:
oversized = EarthLens(
    data_source="overture",
    dataset="buildings",
    variables=[],
    aoi=[-180, -90, 180, 90],
    path=OUT,
)

Calling `download()` on the whole-Earth request trips the cap and raises:

In [ ]:
try:
    oversized.download()
except ValueError as exc:
    print(exc)